# 04 · B1 protocol audit · B2 main benchmark · B6 replication
Nine models × three protocols × two tasks × L ∈ {1, best L} × seeds, on **every** configured dataset.

* **S-R raw** — stratified random split, duplicates kept (the usual protocol in the literature)
* **S-R dedup** — stratified random split after deduplication
* **S-C** — per-class chronological split with embargo (**the main benchmark, B2**)

**H1:** random splits on non-deduplicated data overstate macro-F1 and change the ranking. Running the same grid on CICIDS2017 (and CSE-CIC-IDS2018) gives the **B6 replication**.

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 200, 'display.max_columns', 60)
from sdnids_bench.config import load_config, mode_params, active_datasets, enable_determinism
cfg = load_config('..')
mp = mode_params(cfg)
enable_determinism()
print('mode:', cfg['mode'], '| results:', cfg['out_dir'])
from sdnids_bench.experiments import run_grid, load_results, choose_best_L, PROTOCOLS
DATASETS = active_datasets(cfg, 'B1')
best_L = choose_best_L(cfg)
Ls = sorted({1, best_L})
print('B1 on:', DATASETS, '| L:', Ls, '| S-C seeds:', mp['seeds'], '| random-split seeds:', mp.get('sr_seeds', mp['seeds']))

Primary dataset first, so its results are available early:

In [ ]:
for ds in DATASETS:
    # main benchmark (S-C) with the full seed set ...
    run_grid(cfg, 'B1', [ds], cfg['tasks'], ['SC'], Ls, mp['seeds'], cfg['models'])
    # ... and the random-split comparison arms (H1); 'reduced' mode runs these with 5 seeds
    run_grid(cfg, 'B1', [ds], cfg['tasks'], ['SR_raw', 'SR_dedup'], Ls, mp.get('sr_seeds', mp['seeds']), cfg['models'])

In [ ]:
res = load_results(cfg, DATASETS)
b1 = res[res.exp=='B1']
display(b1.pivot_table(index=['dataset','task','L','model'], columns='protocol', values='macro_f1', aggfunc='mean').round(4))